In [1]:
import os
import sys

project_root = os.path.abspath("..")   # if the notebook is inside notebooks/
sys.path.insert(0, project_root)
print(project_root)

c:\Projects\Service Event Framework\Improvement\code


In [2]:
import os
import numpy as np
import pandas as pd
from utils.dbconnection import engine
from utils.config import config

2026-09-17 15:24:16,686 - Logger: Log file: c:\Projects\Service Event Framework\Improvement\code\logs\SEF_Improvement_20260917_152416.log
2026-09-17 15:24:16,702 - SQLAlchemy: Database connection configured successfully!


In [3]:
from utils.config import config

In [4]:
file_name = 'ServiceEvent_Material.sql'
sql_path = os.path.join(config.locations.queries_folder, file_name)
df = pd.read_sql_query(open(sql_path, "r").read(), engine)
df.head()

,DServiceEventID,Serviceeventcode,ServiceEventName,Priority,confidence,Itemnumber,MaterialName,MaterialGroup,MaterialGroupDescription
0,1,BladeBearingReplace,Blade Bearing Replacement,9.6,10.0,107033,BLADE BEARING ø54,31,"Bearings, Blade TRAC"
1,1,BladeBearingReplace,Blade Bearing Replacement,9.6,10.0,760306,BLADE BEARING MODULE,327,Nacelle modules and
2,1,BladeBearingReplace,Blade Bearing Replacement,9.6,10.0,76221907,BLADE BEARING MODULE,30,Bearings
3,1,BladeBearingReplace,Blade Bearing Replacement,9.6,10.0,76221971,BLADE BEARING MODULE,327,Nacelle modules and
4,4,GenBearReplace,Generator Bearing Replacement,7.8,10.0,103566,"BEARING,BALL,6344 MC3,220 mm",30,Bearings


In [5]:
file_name = 'ServiceEvent_MaterialGroup.sql'
sql_path = os.path.join(config.locations.queries_folder, file_name)
qw = pd.read_sql_query(open(sql_path, "r").read(), engine)
qw.head()

,DServiceEventID,Serviceeventcode,ServiceEventName,Priority,confidence,MaterialGroup,MaterialGroupDescription
0,262,SkyLightCoverReplace,Sky Light Cover replacement,5.16,5.0,105,Covers
1,3,GearReplace,Gearbox,10.00,10.0,211,Gearboxes not mounte
2,631,BladeStudAssemblyReplace,Blade Stud Assembly Replace,7.07,6.0,467,Site Parts Blade/Hub
3,6,GenReplace,Generator Replacement,9.10,10.0,221,Generators not mount
4,12,YawGearReplace,Yaw Gear Replacement,7.47,7.0,218,"Gears, yaw TRACE"


1. Materials being present in multiple events

In [6]:
df1 = df[['Serviceeventcode', 'Itemnumber']].drop_duplicates()
df1.shape

(8109, 2)

In [7]:
df1c = pd.DataFrame(df1['Itemnumber'].value_counts()).reset_index()
df1c = df1c.loc[df1c['count']>1]
df1c.shape

(27, 2)

In [8]:
df1c['Itemnumber'].unique()

array(['29097212', '753257', '114447', '753264', '753889', '753770',
       '29023724', '753780', '753256', '29020336', '153828', '29080472',
       '29087200', '29133149', '29133175', '29153440', '29153441',
       '29332819', '29332860', '29333356', '29333364', '29333367',
       '29333368', '29349561', '29349658', '29452248', '29204214'],
      dtype=object)

In [9]:
df1m = df1.merge(df1c, on='Itemnumber', how='inner')
df1m.sort_values(by='Itemnumber', inplace=True)
df1m.head()

,Serviceeventcode,Itemnumber,count
30,ConverterThermostatReplace,114447,2
3,ThermoValveCACReplace,114447,2
14,BladeStudAssemblyReplace,153828,2
12,YawClawBeamBoltReplace,153828,2
10,YawClawBeamModuleReplace,29020336,2


2. Material Groups present in multiple events

In [10]:
qw1 = qw[['Serviceeventcode', 'MaterialGroup']]
qw1 = qw1.drop_duplicates()
qw1.shape

(47, 2)

In [11]:
qw1c = pd.DataFrame(qw1['MaterialGroup'].value_counts()).reset_index()
qw1c = qw1c.loc[qw1c['count']>1]
qw1c.shape, qw1c['MaterialGroup'].unique()

((4, 2), array(['75', '490', '523', '255'], dtype=object))

In [12]:
qw1m = qw1.merge(qw1c, on='MaterialGroup', how='inner')
qw1m.sort_values(by='MaterialGroup', inplace=True)
qw1m

,Serviceeventcode,MaterialGroup,count
0,RTMHosesReplace,255,2
4,HydCoolPipeReplace,255,2
3,NoseRepairKit_mat,490,2
9,SpinnerNoseConeReplace,490,2
1,FlyWheel_mat,523,2
5,TowerServiceLiftReplace,523,2
2,PPRdcCapReplace,75,4
6,PPRacCapReplace,75,4
7,CACdcCapReplace,75,4
8,PPRTrafoCapReplace,75,4


3. Material being present in one service event but the respective material group in a different service event

In [13]:
req_cols = ['Serviceeventcode', 'Itemnumber', 'MaterialName', 'MaterialGroup', 'MaterialGroupDescription']
mg_all  = df[req_cols].drop_duplicates()
mg_all.shape

(8132, 5)

In [14]:
mg1 = mg_all.merge(qw1, on='MaterialGroup', how='inner', suffixes=('_primary', '_secondary'))
mg1

,Serviceeventcode_primary,Itemnumber,MaterialName,MaterialGroup,MaterialGroupDescription,Serviceeventcode_secondary
0,BladeBearingReplace,107033,BLADE BEARING ø54,31,"Bearings, Blade TRAC",BladeBearingReplace
1,MainBearingReplace,29155641,MAIN SHAFT SER. ASSY 4MW MK 3E,455,"Shafts, main",MainBearingReplace
2,MainBearingReplace,29195008,PTR & AC GEN 5.6MW LTQ,540,Transmission modules,PowerTrainAssembly_mat
3,PitchCylinderReplace,108613,"CYLINDER,HYDRAULIC,125 mm,80 mm,922 mm",251,"Hy cylinders, pitch",PitchCylinderReplace
4,PitchCylinderReplace,108614,"CYLINDER,HYDRAULIC,125 mm,80 mm,922 mm",251,"Hy cylinders, pitch",PitchCylinderReplace
...,...,...,...,...,...,...
1675,HydOilHoseReplace,781673,"HOSE ASSY,HYD,9.525 mm,2325 mm,330 bar",254,Hydraulic hoses,HydOilHoseReplace
1676,HydOilHoseReplace,781674,"HOSE ASSY,HYD,6.35 mm,2105 mm,400 bar",254,Hydraulic hoses,HydOilHoseReplace
1677,HydOilHoseReplace,782038,"HOSE ASSY,HYD,19.05 mm,3760 mm,248 bar",254,Hydraulic hoses,HydOilHoseReplace
1678,HydOilHoseReplace,788931,"HOSE ASSY,HYD,12.7 mm,1580 mm,275 bar",254,Hydraulic hoses,HydOilHoseReplace


In [15]:
mismatches = mg1.loc[mg1['Serviceeventcode_primary'] != mg1['Serviceeventcode_secondary']]
mismatches

,Serviceeventcode_primary,Itemnumber,MaterialName,MaterialGroup,MaterialGroupDescription,Serviceeventcode_secondary
2,MainBearingReplace,29195008,PTR & AC GEN 5.6MW LTQ,540,Transmission modules,PowerTrainAssembly_mat
24,GearHSS,29102035,ADA REPAIR KIT ZFR916011454 GE,383,Power module,PPRSkiipReplace
25,GearHSS,789405,EF901E-404L KIT STAGE 3 - 60Hz,211,Gearboxes not mounte,GearReplace
26,GenFanReplace,70000506,COOLING FAN - ICDS,230,Heat exchangers,CACHeatExchangerMGReplace
27,CACPumpReplace,101011,COOLING UNIT V39 OIL COOLER,230,Heat exchangers,CACHeatExchangerMGReplace
...,...,...,...,...,...,...
1654,HydOilHoseReplace,763573,PIPE FOR PITCH CYLINDER,255,Hydr.-lubr. & cool.,HydCoolPipeReplace
1655,HydOilHoseReplace,763612,"HOSE ASSY,HYD,25.4 mm,1082 mm,260 bar",255,Hydr.-lubr. & cool.,RTMHosesReplace
1656,HydOilHoseReplace,763612,"HOSE ASSY,HYD,25.4 mm,1082 mm,260 bar",255,Hydr.-lubr. & cool.,HydCoolPipeReplace
1657,HydOilHoseReplace,763651,"HOSE ASSY,HYD,25.4 mm,1454 mm,260 bar",255,Hydr.-lubr. & cool.,RTMHosesReplace


In [16]:
# validate mismatches: Case 1
df.loc[df['Itemnumber']=='29195008']

,DServiceEventID,Serviceeventcode,ServiceEventName,Priority,confidence,Itemnumber,MaterialName,MaterialGroup,MaterialGroupDescription
86,7,MainBearingReplace,Main Bearing Arrangement Replacement,9.0,10.0,29195008,PTR & AC GEN 5.6MW LTQ,540,Transmission modules


In [17]:
qw.loc[qw['MaterialGroup']=='540']

,DServiceEventID,Serviceeventcode,ServiceEventName,Priority,confidence,MaterialGroup,MaterialGroupDescription
6,956,PowerTrainAssembly_mat,Power Train Assembly Replace,9.2,10.0,540,Transmission modules


In [18]:
# validate mismatches: Case 2
df.loc[df['Itemnumber']=='29102035']

,DServiceEventID,Serviceeventcode,ServiceEventName,Priority,confidence,Itemnumber,MaterialName,MaterialGroup,MaterialGroupDescription
358,18,GearHSS,GearBox HSS Stage,7.72,10.0,29102035,ADA REPAIR KIT ZFR916011454 GE,383,Power module


In [19]:
qw.loc[qw['MaterialGroup']=='383']

,DServiceEventID,Serviceeventcode,ServiceEventName,Priority,confidence,MaterialGroup,MaterialGroupDescription
7,97,PPRSkiipReplace,PPR SKiiPack Replacement,7.03,6.0,383,Power module


4. Materials belonging to same material group are in different Service Events

In [20]:
df.head()

,DServiceEventID,Serviceeventcode,ServiceEventName,Priority,confidence,Itemnumber,MaterialName,MaterialGroup,MaterialGroupDescription
0,1,BladeBearingReplace,Blade Bearing Replacement,9.6,10.0,107033,BLADE BEARING ø54,31,"Bearings, Blade TRAC"
1,1,BladeBearingReplace,Blade Bearing Replacement,9.6,10.0,760306,BLADE BEARING MODULE,327,Nacelle modules and
2,1,BladeBearingReplace,Blade Bearing Replacement,9.6,10.0,76221907,BLADE BEARING MODULE,30,Bearings
3,1,BladeBearingReplace,Blade Bearing Replacement,9.6,10.0,76221971,BLADE BEARING MODULE,327,Nacelle modules and
4,4,GenBearReplace,Generator Bearing Replacement,7.8,10.0,103566,"BEARING,BALL,6344 MC3,220 mm",30,Bearings


In [21]:
req_cols = ['Serviceeventcode', 'MaterialGroup']
dmg_all = df[req_cols].drop_duplicates()
dmg_all.head()

,Serviceeventcode,MaterialGroup
0,BladeBearingReplace,31
1,BladeBearingReplace,327
2,BladeBearingReplace,30
4,GenBearReplace,30
6,GenBearReplace,222


In [22]:
dmgg = dmg_all.groupby('MaterialGroup').size().reset_index(name='count')
dmgg.sort_values(by='count', ascending=False, inplace=True)
dmgg.head()

,MaterialGroup,count
14,175,51
89,445,39
50,275,33
47,255,32
127,65,29


In [23]:
dmgg2 = dmgg.loc[dmgg['count']==2]
dmgg2 = df.loc[df['MaterialGroup'].isin(dmgg2['MaterialGroup'])]
dmgg2 = dmgg2.sort_values(by='MaterialGroup')
req_cols = ['Serviceeventcode', 'MaterialGroup', 'MaterialGroupDescription']
dmgg2f = dmgg2[req_cols].drop_duplicates()
dmgg2f.head(10)

,Serviceeventcode,MaterialGroup,MaterialGroupDescription
2437,SkyLightReplace,105,Covers
6778,NacelleBottomCoverRepair,105,Covers
7455,BladeLightReceptorScrew_mat,145,Earthing equipment
4138,DownConductorReceptorRepair,145,Earthing equipment
3514,GroundCntrlReplace,229,Ground controllers
3069,ControllerCabinetFanReplace,229,Ground controllers
2373,PPRHVCableReplace,248,High voltage cables
2807,ConvCableReplace,248,High voltage cables
1481,PPRSkiipReplace,383,Power module
358,GearHSS,383,Power module


In [24]:
def get_n_replicates(dmgg, df, n=2):
    dmgg_n = dmgg.loc[dmgg['count']==n]
    dmgg_n = df.loc[df['MaterialGroup'].isin(dmgg_n['MaterialGroup'])]
    dmgg_n = dmgg_n.sort_values(by='MaterialGroup')
    req_cols = ['Serviceeventcode', 'MaterialGroup', 'MaterialGroupDescription']
    dmgg_nf = dmgg_n[req_cols].drop_duplicates()
    return dmgg_nf

In [32]:
get_n_replicates(dmgg, df, 3).shape

(48, 3)

Save results

In [33]:
df_save_dict = {
    'event_material': df,
    'event_materialGroup': qw,
    'material_duplicacy': df1m,
    'materialGroup_duplicacy': qw1m,
    'material_vs_materialGroup': mismatches,
    'materialgroup_nEvents_list': dmgg2f,
    'materialgroup_nEvents': dmgg2
}

In [34]:
from pathlib import Path

output_dir = Path(config.locations.outputs)
output_dir.mkdir(parents=True, exist_ok=True)
excel_path = output_dir / "service_event_analysis.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    for sheet_name, df_sheet in df_save_dict.items():
        if isinstance(df_sheet, pd.DataFrame):
            df_sheet.to_excel(writer, sheet_name=sheet_name, index=False)
        else:
            pd.DataFrame(df_sheet).to_excel(writer, sheet_name=sheet_name, index=False)

excel_path

WindowsPath('C:/Projects/Service Event Framework/Improvement/code/data/outputs/service_event_analysis.xlsx')